<a href="https://colab.research.google.com/github/tazir-shaif/ai-engineering-portfolio/blob/main/module-6-evaluation-monitoring/Module_6_Session_3_Golden_Set_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 6 — Session 3: Golden Set Evaluation

## What we're building
So far we've monitored live traffic with LangSmith and Opik.
But monitoring only tells you what happened AFTER deployment.
What if you want to test BEFORE you deploy a new prompt or model?

A Golden Set is a curated collection of question-answer pairs
where you already know what the correct answer should look like.
You run your LLM against this set and measure how well it performs.
It's like a exam with an answer key — you know the right answers,
so you can grade the LLM objectively.

## Real world analogy
Swiggy's QA team has 50 standard customer complaints they test
every new support bot version against before releasing it.
If the bot scores below 8/10 on those 50 complaints, it doesn't
get deployed. That's a Golden Set in practice.

## AWS equivalent
Amazon SageMaker Model Monitor + SageMaker Clarify +
Bedrock Model Evaluation with curated test datasets

## Step 0: Install libraries
Same libraries as previous sessions plus one new one:
- langchain-groq: our Groq wrapper (familiar)
- opik: for evaluation framework (familiar)
- pandas: for building and displaying our golden set as a table (familiar)

In [ ]:
!pip install -q langchain-groq opik pandas
print("✅ Libraries installed")

## Step 1: Build the Golden Set
A golden set has three columns:
- input: the customer complaint (the question)
- expected_output: what a perfect answer looks like (the answer key)
- context: any background info the LLM should know (optional)

We build 5 Swiggy complaints covering different issue types.
In production, a golden set has 50-500 examples.
For our portfolio, 5 is enough to demonstrate the concept.

In [ ]:
import os
import pandas as pd
from google.colab import userdata
from langchain_groq import ChatGroq

# Load keys
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
os.environ["OPIK_API_KEY"] = userdata.get("OPIK_API_KEY")

# Set up LLM
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0.3)

# --- Build the Golden Set ---
# Each entry has: input (complaint), expected_output (ideal answer), context (policy)
golden_set = [
    {
        "input": "My Swiggy order is 45 minutes late and the food is cold. I want a refund.",
        "expected_output": "Apologise for the delay, offer full refund, ask for order ID to process it.",
        "context": "Swiggy policy: full refund for orders more than 30 minutes late."
    },
    {
        "input": "I received the wrong items in my Zepto order. I ordered biryani but got pizza.",
        "expected_output": "Apologise for wrong order, offer replacement or full refund, ask for order ID.",
        "context": "Swiggy policy: replacement or full refund for wrong items delivered."
    },
    {
        "input": "My Swiggy order was marked delivered but I never received it.",
        "expected_output": "Apologise sincerely, escalate to investigation team, offer full refund immediately.",
        "context": "Swiggy policy: immediate refund for orders marked delivered but not received."
    },
    {
        "input": "I found a hair in my food from Swiggy. This is disgusting.",
        "expected_output": "Apologise sincerely for hygiene issue, offer full refund, escalate to restaurant quality team.",
        "context": "Swiggy policy: full refund and restaurant quality flag for hygiene complaints."
    },
    {
        "input": "Swiggy charged me twice for the same order. I can see two debits on my bank statement.",
        "expected_output": "Apologise for double charge, assure refund of duplicate charge within 5-7 business days, ask for order ID.",
        "context": "Swiggy policy: duplicate charges refunded within 5-7 business days."
    },
]

# Convert to DataFrame so it looks clean
df = pd.DataFrame(golden_set)
print("✅ Golden Set built —", len(df), "examples")
print("\nComplaint types covered:")
for i, row in df.iterrows():
    print(f"  {i+1}. {row['input'][:60]}...")

## Step 2: Run LLM on every complaint in the Golden Set
We send each complaint to the LLM and collect its actual response.
Then we'll compare actual vs expected — that's the evaluation.
Think of it as giving the LLM the exam questions and collecting its answer sheet.

In [ ]:
# Run the LLM on every complaint and collect actual responses
# We store results in a new column: "actual_output"

actual_outputs = []                              # empty list to collect replies

for i, row in df.iterrows():                     # loop through each row in golden set
    # Build a prompt that includes the policy context
    prompt = f"""You are a Swiggy customer support agent.

Policy context: {row['context']}

Customer complaint: {row['input']}

Write a short, empathetic support reply."""

    # Send to LLM
    response = llm.invoke(prompt)
    actual_outputs.append(response.content)      # save the reply

    # Print progress so we know it's working
    print(f"✅ Complaint {i+1} answered")

# Add actual outputs as a new column in our DataFrame
df["actual_output"] = actual_outputs

print("\n✅ All complaints answered by LLM")
print("DataFrame now has columns:", list(df.columns))

## Step 3: Score each answer using LLM-as-Judge
We compare actual_output vs expected_output for each complaint.
The judge scores on three things:
- Did it apologise? (empathy)
- Did it offer the right resolution? (correctness)
- Did it ask for order ID? (completeness)

Score: 0.0 to 1.0 for each complaint.
This tells us: where does our bot do well, and where does it fail?

In [ ]:
import json

def score_response(complaint, expected, actual, context):
    """
    LLM-as-Judge: scores actual response vs expected response.
    Returns a score (0.0-1.0) and a reason.
    """
    judge_prompt = f"""You are an expert Swiggy customer support evaluator.

Compare the ACTUAL response against the EXPECTED response criteria.

Customer complaint: {complaint}
Policy context: {context}

EXPECTED response should: {expected}

ACTUAL response given: {actual}

Score the ACTUAL response from 0.0 to 1.0 based on:
- 1.0 = fully meets expected criteria (apologised + correct resolution + asked for order ID)
- 0.7 = mostly meets criteria but missing one element
- 0.5 = partially meets criteria, missing key elements
- 0.3 = poor response, mostly off target
- 0.0 = completely wrong or inappropriate

Reply ONLY with a JSON object like this:
{{"score": 0.9, "reason": "your reason here"}}"""

    response = llm.invoke(judge_prompt)

    # Parse the JSON response
    try:
        # Clean any markdown formatting the LLM might add
        clean = response.content.strip().replace("```json", "").replace("```", "")
        result = json.loads(clean)
        return result["score"], result["reason"]
    except:
        # If JSON parsing fails, return a default
        return 0.5, "Could not parse judge response"

print("✅ Judge function defined")
print("Ready to score all 5 complaints")

## Step 4: Run the judge on all 5 complaints
We score every actual response against the expected criteria.
This gives us a score per complaint — our evaluation report.

In [ ]:
# Run the judge on every complaint and collect scores
scores = []
reasons = []

for i, row in df.iterrows():
    score, reason = score_response(
        complaint=row["input"],
        expected=row["expected_output"],
        actual=row["actual_output"],
        context=row["context"]
    )
    scores.append(score)
    reasons.append(reason)
    print(f"Complaint {i+1}: score = {score}")

# Add scores and reasons to DataFrame
df["score"] = scores
df["reason"] = reasons

# Calculate overall performance
avg_score = df["score"].mean()
min_score = df["score"].min()
max_score = df["score"].max()

print("\n=== Golden Set Evaluation Results ===")
print(f"Average score:  {avg_score:.2f}")
print(f"Highest score:  {max_score:.2f}")
print(f"Lowest score:   {min_score:.2f}")
print(f"Pass rate (>=0.7): {(df['score'] >= 0.7).sum()}/5 complaints")

## Step 5: Detailed Results Table
Numbers alone don't tell the full story.
We need to see the reasons to know HOW to improve the bot.
This is the debugging step — find the failure patterns.

In [ ]:
# Print detailed results for each complaint
print("=== DETAILED EVALUATION REPORT ===\n")

for i, row in df.iterrows():
    print(f"--- Complaint {i+1} ---")
    print(f"Issue:    {row['input'][:70]}...")
    print(f"Score:    {row['score']}")
    print(f"Reason:   {row['reason']}")
    print(f"Expected: {row['expected_output']}")
    print(f"Actual:   {row['actual_output'][:150]}...")
    print()

# Show a clean summary table
print("=== SUMMARY TABLE ===")
summary = df[["input", "score", "reason"]].copy()
summary["input"] = summary["input"].str[:50] + "..."   # truncate for display
print(summary.to_string(index=False))

## Step 6: Fix the prompt and re-evaluate
The golden set revealed a systematic weakness:
the bot never asks for the order ID.
We fix the system prompt and re-run the evaluation.
This is the prompt improvement loop — the core of LLM engineering.
Fix → Evaluate → Compare → Repeat.

In [ ]:
# --- IMPROVED PROMPT with explicit order ID instruction ---

def get_improved_response(complaint, context):
    """Improved prompt that explicitly instructs the bot to ask for order ID"""
    prompt = f"""You are a Swiggy customer support agent.

Policy context: {context}

IMPORTANT RULES — follow all of these in every response:
1. Always apologise sincerely in the first sentence
2. Always offer the correct resolution based on policy
3. Always ask for the order ID to process the request
4. For missing deliveries — always mention escalation to investigation team
5. Keep the response under 100 words

Customer complaint: {complaint}

Write a short, empathetic support reply following ALL rules above."""

    response = llm.invoke(prompt)
    return response.content

# Run improved prompt on all 5 complaints
improved_outputs = []

for i, row in df.iterrows():
    improved = get_improved_response(row["input"], row["context"])
    improved_outputs.append(improved)
    print(f"✅ Complaint {i+1} answered with improved prompt")

df["improved_output"] = improved_outputs
print("\n✅ All improved responses collected")

## Step 7: Re-evaluate with improved prompt
We run the same judge on the improved responses.
Then we compare old scores vs new scores side by side.
This is the ablation study — did our fix actually help?

In [ ]:
# Score the improved responses using the same judge function
improved_scores = []
improved_reasons = []

for i, row in df.iterrows():
    score, reason = score_response(
        complaint=row["input"],
        expected=row["expected_output"],
        actual=row["improved_output"],    # ← improved output this time
        context=row["context"]
    )
    improved_scores.append(score)
    improved_reasons.append(reason)
    print(f"Complaint {i+1}: old score = {row['score']} → new score = {score}")

# Add to DataFrame
df["improved_score"] = improved_scores
df["improved_reason"] = improved_reasons

# Compare old vs new
print("\n=== BEFORE vs AFTER PROMPT FIX ===")
print(f"{'Complaint':<12} {'Before':>8} {'After':>8} {'Change':>8}")
print("-" * 40)
for i, row in df.iterrows():
    change = row["improved_score"] - row["score"]
    arrow = "⬆️" if change > 0 else ("⬇️" if change < 0 else "➡️")
    print(f"Complaint {i+1}  {row['score']:>8.1f} {row['improved_score']:>8.1f}  {arrow} {change:+.1f}")

print("-" * 40)
print(f"{'Average':<12} {df['score'].mean():>8.2f} {df['improved_score'].mean():>8.2f}  {'':>4} {df['improved_score'].mean() - df['score'].mean():+.2f}")
print(f"\nOld pass rate (>=0.7): {(df['score'] >= 0.7).sum()}/5")
print(f"New pass rate (>=0.7): {(df['improved_score'] >= 0.7).sum()}/5")

## Step 8: Save Evaluation Report
We save the full results to CSV — inputs, expected outputs,
actual outputs, improved outputs, and all scores.
This is your evidence file — proof that your evaluation worked.
In production this would go to S3 and trigger a CloudWatch alert
if average score drops below the threshold.

In [ ]:
# Save the full evaluation report to CSV
report_cols = [
    "input",           # the complaint
    "expected_output", # what a perfect answer looks like
    "actual_output",   # what the original prompt produced
    "score",           # original score
    "reason",          # why it got that score
    "improved_output", # what the improved prompt produced
    "improved_score",  # improved score
    "improved_reason"  # why it got the new score
]

report = df[report_cols].copy()
report.to_csv("golden_set_evaluation_report.csv", index=False)

print("✅ Evaluation report saved to golden_set_evaluation_report.csv")
print(f"Rows: {len(report)}")
print(f"Columns: {list(report.columns)}")
print(f"\nFinal Summary:")
print(f"  Original average score:  {df['score'].mean():.2f}")
print(f"  Improved average score:  {df['improved_score'].mean():.2f}")
print(f"  Improvement:             +{df['improved_score'].mean() - df['score'].mean():.2f}")
print(f"  Original pass rate:      {(df['score'] >= 0.7).sum()}/5")
print(f"  Improved pass rate:      {(df['improved_score'] >= 0.7).sum()}/5")

## Session Summary

### What we built
A complete Golden Set Evaluation pipeline for a Swiggy support bot:

1. Built a 5-example golden set covering 5 different complaint types:
   - Late + cold delivery
   - Wrong items
   - Missing delivery (most serious)
   - Hygiene complaint
   - Double charge

2. Ran original prompt on all 5 complaints → average score 0.72

3. Used LLM-as-Judge to score each response against expected criteria

4. Discovered systematic weakness: bot never asked for order ID
   (affected 4/5 complaints)

5. Fixed the prompt with explicit rules:
   - Always ask for order ID
   - Always escalate missing deliveries to investigation team

6. Re-evaluated with improved prompt → average score 1.00 (+0.28)

7. Saved full evaluation report to CSV

### The Golden Set loop
Build golden set → Run LLM → Score → Find pattern → Fix → Re-score → Compare
This loop is what separates junior from senior AI engineers.
Junior engineers guess at improvements. Senior engineers measure them.

### Key concepts
- Golden Set: curated Q&A pairs with known correct answers — the exam answer key
- LLM-as-Judge: uses an LLM to score another LLM's output (0.0 to 1.0)
- Pass threshold: 0.7 minimum score to consider a response acceptable
- Ablation study: comparing before vs after a single change to measure its impact
- Systematic weakness: a failure pattern that appears across multiple test cases

### Why this matters for FAANG interviews
"How do you know your prompt is good?"
Answer: I run it against a golden set of curated examples,
score with LLM-as-Judge, and measure pass rate.
If pass rate drops below threshold, I don't deploy.

### AWS equivalent
- Golden Set storage → Amazon S3
- Evaluation pipeline → AWS Lambda + Step Functions
- Score tracking → Amazon CloudWatch custom metrics
- Threshold alerting → CloudWatch Alarms → SNS notification
- Full managed version → Amazon Bedrock Model Evaluation

### What is next
Module 6 Session 4 — End to End Evaluation Pipeline:
Combining LangSmith tracing + Opik monitoring + Golden Set evaluation
into one unified pipeline that runs automatically.